<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/rr_respiratory_mortality/notebooks/02.RR_Respiratory_Mortality_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#📊 Exploratory Analysis of Relative Risk of Respiratory Mortality Attributable to Long-Term PM₂.₅ Exposure in Argentina (2022)

In this notebook we will perform an exploratory analysis of Relative Risk (RR) of Respiratory Mortality Attributable to Long-Term PM₂.₅ Exposure in Argentina for the year 2022. The dataset integrates model-based predictions of asthma mortality derived from a Random Forest algorithm with epidemiologically derived estimates of RR of respiratory mortality, expressed through lower, central, and upper bounds. [1] These estimates reflect uncertainty in long-term exposure to ambient PM₂.₅ and its contribution to respiratory-related mortality, making rigorous data exploration a critical prerequisite for subsequent analytical stages.

The main goal of this exploratory phase is to assess data quality, characterize the statistical properties of the  RR of respiratory mortality, and identify structural features such as skewness, dispersion, and potential outliers. Through  data inspection, summary statistics, and visualization of distributions, this notebook establishes an empirical foundation for downstream correlation analysis, spatial modeling, and inferential techniques. Particular attention is paid to understanding variability across departments and uncertainty intervals, ensuring that later modeling decisions are informed by the underlying behavior of the data rather than implicit assumptions.

## 🤖 Load libraries

The libraries required for the analysis will be loaded

In [ ]:
# dataframe libraries
import pandas as pd

# Import NumPy for numerical and mathematical operations
import numpy as np

# geospatial libraries
import geopandas as gpd

# plot libraries
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator

# other libraries
import os
from itables import init_notebook_mode, show

In [ ]:
# change directory to work folder (at the begining docker container enter into /home/jovyan/)
%cd work

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

##🗂 Load and prepare the dataset

In [ ]:
# open gdf data
gdf_map_rr = gpd.read_file("pdt/rr_respiratory_mortality/data/gpkg/gdf_rrrm.gpkg")

In [ ]:
# visualize geodataframe
init_notebook_mode(all_interactive=True)
gdf_map_rr

In [ ]:
# visualize geodataframe
init_notebook_mode(all_interactive=True)
gdf_map_rr

In [ ]:
# visualize column names
gdf_map_rr.columns

In [ ]:
# select 'IDDPTO', 'AMR_PRED_2022' 'RRRM_2022_L' 'RRRM_2022_C', 'RRRM_2022_U' columns
df = gdf_map_rr[[ 'IDDPTO', 'AMR_PRED_2022', 'RRRM_2022_L', 'RRRM_2022_C', 'RRRM_2022_U']]

In [ ]:
# visualize daf
init_notebook_mode(all_interactive=True)
df

In [ ]:
# original length of data
len(df)

In [ ]:
# drop all rows containing NaN for all rows
df = df.dropna()

In [ ]:
# length of data after dropping NaN
len(df)

In [ ]:
# visualize df
init_notebook_mode(all_interactive=True)
df

## 🧐Data Inspection

In this step, the goal is to develop a first understanding of the dataset’s structure and integrity before performing any statistical or graphical analysis. Data inspection is a critical quality-control phase in epidemiological and data science workflows, as it helps detect missing values, inconsistent data types, or structural anomalies that could bias future analyses such as correlation or spatial modeling.

In [ ]:
# Display the dimensions of the dataset
# The output shows (number_of_rows, number_of_columns)
df.shape

In [ ]:
# Display detailed information about the DataFrame
# Includes column names, data types, non-null counts, and memory usage
df.info()

In [ ]:
# Check for missing values in each column
# isnull() returns a boolean DataFrame indicating missing values
# sum() counts the number of missing values per column
init_notebook_mode(all_interactive=True)
df.isnull().sum()

The results indicate that the dataset contains no missing values, inconsistent data types, or structural anomalies.

## 🎯 Statistical Summaries

This step focuses on quantitatively characterizing the central tendency, dispersion, and range of the RR of respiratory mortality the predicted asthma mortality rate. Statistical summaries provide a description of the data, allowing us to identify scale differences, variability, skewness proxies, and potential inconsistencies across variables before conducting correlation or modeling analyses.

In [ ]:
# Select only the numerical variables relevant for statistical analysis
numerical_vars = [
    "AMR_PRED_2022",
    "RRRM_2022_L",
    "RRRM_2022_C",
    "RRRM_2022_U"
]

In [ ]:
# Create a new DataFrame containing only the selected numerical variables
df_numeric = df[numerical_vars]

In [ ]:
# Generate descriptive statistics for the numerical variables
# This includes count, mean, standard deviation, min, max, and quartiles
init_notebook_mode(all_interactive=True)
df_numeric.describe()

In [ ]:
# calculate  Coefficient of Variation (CV)
cv_series = (df_numeric.std() / df_numeric.mean()) * 100
print("Coefficient of Variation (%):")
print(cv_series)

Across the 170 departments, the predicted asthma mortality rate is  more heterogeneous than the respiratory mortality RR. The predicted mortality rate exhibits a  higher absolute standard deviation (≈2.33 vs. ≈0.04 for the central RR bound), indicating greater dispersion, and a coefficient of variation nearly 19 times larger (CV = ≈ 72.22 vs. ≈4.25 for the central RR bound). This demonstrates that asthma mortality rates vary far more across departments relative to their mean than the RR values.

In [ ]:
# Calculate the variance for each numerical variable
# Variance provides another measure of dispersion
# Variance answer the question "How far, on average, do observations deviate from the mean in absolute terms?"
init_notebook_mode(all_interactive=True)
df_numeric.var()


Statistical variance also confirm that the predicted asthma mortality rate exhibits  high  variability (≈ 5.72), indicating  more heterogeneity and strong concentration of mortality in a limited number of departments. In contrast, RRs of respiratory mortality  show low variability, particularly for the lower bound, reflecting their generally small magnitudes and constrained dispersion across most departments.

## 📈 Data Distribution

In this step, we examine the  distribution of the numerical variables to understand their shape, skewness, and concentration. Visualizing distributions is essential before correlation or modeling because many statistical methods implicitly assume approximate symmetry or are sensitive to heavy tails and zero inflation—both common in epidemiological rate data.

In [ ]:
# Define the numerical variables to visualize
numerical_vars = [
    "AMR_PRED_2022",
    "RRRM_2022_L",
    "RRRM_2022_C",
    "RRRM_2022_U"
]

In [ ]:
# Set the number of bins for all histograms
# Using a consistent bin count allows comparison across variables
bins = 30

In [ ]:
# Create a single figure with a 2x2 grid of subplots
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8), gridspec_kw={'hspace': 0.5})

# Flatten the 2D array of axes into a 1D array for easy iteration
axes = axes.flatten()

# set histrogram titles
titles_hist = [
    "Distribution of Predicted \nAsthma Mortality Rate",
    "Distribution of  Respiratory Mortality \nRR Lower Bound",
    "Distribution of  Respiratory Mortality \nRR Central Bound",
    "Distribution of  Respiratory Mortality \nRRUpper Bound"
    ]

# set histogram X-axis label
xlabels_hist = [
    "Deaths per\n100,000 population",
    "Relative Risk",
    "Relative Risk",
    "Relative Risk"
    ]

# Loop over variables and corresponding subplot axes
for ax, title, xlabel, var in zip(axes, titles_hist, xlabels_hist, numerical_vars):

    # Plot the histogram for the current variable on the assigned axis
    # dropna() ensures missing values do not interfere with the plot
    ax.hist(df[var].dropna(), bins=bins)

    # Set the title for each subplot
    ax.set_title(f"{title}")

    # Label the x-axis with generic variable name
    ax.set_xlabel(xlabel)

    # Label the y-axis as frequency
    ax.set_ylabel("Frequency")


# Display the figure
plt.show()


The predicted asthma mortality rate exhibits a clear right-skewed distribution, with most departments concentrated at low-to-moderate rates and a long upper tail driven by a small number of high-mortality departments, indicating strong spatial heterogeneity.

All distributions of respiratory mortality RR for the lower, central, and upper bounds are right-skewed and zero-inflated, with most departments clustered approximately between 1.0 and 1.05 and progressively wider tails from the lower to the upper bound. This pattern indicates generally low relative risk in most areas, accompanied by increasing uncertainty and a concentration of higher RR values in a small number of departments.

## 🎚️Handling Outliers

In this step, we will identify and visualize potential outliers in the predicted asthma mortality rate  and the RR of respiratory mortality variables. Outliers are particularly important in our context because a small number of departments with extreme values can influence correlations and inflate variance.

In [ ]:
# Define the numerical variables to analyze for outliers
numerical_vars = [
    "AMR_PRED_2022",
    "RRRM_2022_L",
    "RRRM_2022_C",
    "RRRM_2022_U"
]

In [ ]:
# set boxplots titles
titles_bp = [
    "Predicted \nAsthma Mortality Rate",
    "Respiratory Mortality \nRR Lower Bound",
    "Respiratory Mortality \nRR Central Bound",
    "Respiratory Mortality \nRRUpper Bound"
    ]

In [ ]:
# set histogram Y-axis label
ylabels_bp = [
    "Deaths per\n100,000 population",
    "Relative Risk",
    "Relative Risk",
    "Relative Risk"
    ]

In [ ]:
# Create a single figure with a 2x2 grid of subplots
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8), gridspec_kw={'hspace': 0.5})

# Flatten the axes array for easier iteration
axes = axes.flatten()

# Loop over variables and corresponding subplot axes
for ax, title, ylabel, var  in zip(axes, titles_hist, ylabels_bp, numerical_vars):

    # Create a boxplot for the current variable
    # dropna() ensures missing values do not affect the plot
    ax.boxplot(df[var].dropna(), vert=True)

    # Set the title for each subplot
    ax.set_title(f"Outlier Detection: {title}")

    # Label the y-axis with generic variable name
    ax.set_ylabel(ylabel)

    # not showing x-axis label
    ax.set_xticks([])
    ax.set_xticklabels([])

    # Add a horizontal grid for better readability
    ax.grid(axis="y", linestyle="--", alpha=0.6)

# Display the figure
plt.show()


In [ ]:
# Define the numerical variables to analyze for outliers
numerical_vars = [
    "AMR_PRED_2022",
    "RRRM_2022_L",
    "RRRM_2022_C",
    "RRRM_2022_U"
]

In [ ]:
# set IQR-based outliers titles
titles_iq = [
    "Predicted Asthma Mortality Rate",
    "Respiratory Mortality RR Lower Bound",
    "Respiratory Mortality RR Central Bound",
    "Respiratory Mortality RRUpper Bound"
    ]

In [ ]:
# Loop through each numerical variable to count IQR-based outliers
for var, title in zip(numerical_vars, titles_iq):

    # Compute first and third quartiles
    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)

    # Compute the interquartile range
    IQR = Q3 - Q1

    # Define lower and upper bounds for outliers
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify outliers
    outliers = df[(df[var] < lower_bound) | (df[var] > upper_bound)]

    # Print the number of outliers detected
    print(f"{title}: {outliers.shape[0]} potential outliers")


The interquartile range (IQR) method for detecting outliers shows that the predicted asthma mortality rate has 7 potential outliers, indicating that a small number of departments with high mortality likely drive the overall  heterogeneity. For the respiratory mortality RR, outliers are predominantly the same for all bounds, suggesting that extreme RR values are concentrated in a subset of departments. These outliers are epidemiologically meaningful rather than anomalous, as they represent concentrated high-risk areas.

## 📚 References

1. WHO Regional Office for Europe, European Centre for Environment and Health (2022). AirQ+: software tool for health risk assessment of air pollution. Bonn (Germany): WHO Regional Office for Europe.
https://www.who.int/europe/tools-and-toolkits/airq---software-tool-for-health-risk-assessment-of-air-pollution
Accessed 01/2026